# Gemini Interactions e agentes do MeuControlePC
Exemplo opcional para Google Colab. A aplicação Windows funciona sem este notebook.
Use o ZIP público desta versão, sem seu `.env`, dados, modelos ou arquivos pessoais.
A cota Gemini API é separada do Colab Pro; somente chamadas confirmadas usam sua conta.
Referências: [Interactions](https://ai.google.dev/gemini-api/docs/interactions-overview?hl=pt-br),
[notebook oficial](https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_managed_agents.ipynb).
Nenhuma célula foi executada nem contém resultados, IDs remotos ou credenciais.


In [ ]:
%pip install "google-genai>=2.9,<3"


## Carregar somente as fontes públicas
Selecione o ZIP `MeuControlePC_v2_10_0_integrado.zip` fornecido com a versão.
Esta célula extrai apenas as fontes `.py` da raiz para uma pasta temporária do Colab.
Não inicia o bot, painel, instaladores, modelos Ollama ou túneis.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import io, os, sys, tempfile, zipfile
uploads = files.upload()
if len(uploads) != 1:
    raise ValueError("Selecione somente o ZIP público da versão.")
arquivo = next(iter(uploads.values()))
if len(arquivo) > 20_000_000:
    raise ValueError("Pacote excede o limite deste exemplo.")
fontes = Path(tempfile.mkdtemp(prefix="meucontrolepc_fontes_"))
with zipfile.ZipFile(io.BytesIO(arquivo)) as pacote:
    candidatos = []
    for item in pacote.infolist():
        nome = PurePosixPath(item.filename)
        if nome.is_absolute() or ".." in nome.parts:
            raise ValueError("Caminho inválido no pacote.")
        if len(nome.parts) == 2 and nome.parts[0] == "MeuControlePC_v2_10_0" and nome.suffix == ".py":
            candidatos.append(item)
    if not candidatos or len(candidatos) > 100 or sum(i.file_size for i in candidatos) > 5_000_000:
        raise ValueError("Fontes da versão não encontradas ou excedem o limite.")
    for item in candidatos:
        (fontes / PurePosixPath(item.filename).name).write_bytes(pacote.read(item))
os.environ["MEUCONTROLEPC_DADOS"] = tempfile.mkdtemp(prefix="meucontrolepc_dados_colab_")
sys.path.insert(0, str(fontes))
import gemini_interactions as gemini
print("Fontes públicas preparadas. Nenhum serviço foi iniciado.")


In [ ]:
from getpass import getpass
chave = getpass("Chave Gemini API (não será exibida): ").strip()
if not chave:
    raise ValueError("Informe uma chave para usar a API.")
os.environ["GEMINI_API_KEY"] = chave
del chave
os.environ["GEMINI_MODEL"] = input("Modelo disponível à sua chave [gemini-3.8-flash]: ").strip() or "gemini-3.8-flash"


## Uma conversa pelo modelo, sem armazenar a interação
O envio seguinte usa `store=False` e pode consumir cota. Só o texto escrito é enviado.


In [ ]:
pedido_modelo = input("Pergunta para o modelo: ").strip()
if input("Digite ENVIAR para usar a API: ").strip() == "ENVIAR":
    print(gemini.responder(pedido_modelo))
else:
    print("Envio cancelado.")


## Tarefa gerenciada em segundo plano
Usa `store=True` e a retenção da API Google. O sandbox Antigravity fica sem rede externa.
Nenhum arquivo local, chave do assistente ou ferramenta do PC é enviado ao sandbox.
Um timeout de criação pode deixar o resultado incerto: confira o AI Studio antes de repetir.


In [ ]:
tarefa = None
objetivo = input("Tarefa para Antigravity: ").strip()
if input("Digite ENVIAR TAREFA para criar uma tarefa remota com cota: ").strip() == "ENVIAR TAREFA":
    tarefa = gemini.criar_tarefa(objetivo, confirmado=True)
    print(tarefa["estado"])
else:
    print("Envio cancelado.")


Atualize manualmente. Reexecutar esta célula consulta o ID recebido; não recria a tarefa.


In [ ]:
if not tarefa:
    raise ValueError("Nenhuma tarefa com ID remoto confirmado nesta sessão.")
tarefa = gemini.consultar_tarefa(tarefa["remote_id"])
print(tarefa["estado"])
print(tarefa["texto"])


## Cancelamento opcional
A API pode devolver um estado ainda em andamento. Confira o estado real.
Para tarefas/resultados persistentes por conversa, use o painel do assistente Windows.
Não publique este notebook com saídas, chaves ou resultados pessoais.


In [ ]:
if tarefa and input("Digite CANCELAR TAREFA para solicitar o cancelamento: ").strip() == "CANCELAR TAREFA":
    tarefa = gemini.cancelar_tarefa(tarefa["remote_id"], confirmado=True)
    print(tarefa["estado"])
